# Beat 3: Variantes de ReLU (Deconvnet e Guided Backpropagation)

**Pergunta:** Por que Deconv e Guided parecem melhores?

Este notebook apresenta a diferença nas regras de retropropagação através das ReLUs (Vanilla, Deconvnet e Guided Backpropagation) e mostra a medição de discriminação de classe entre os métodos.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('notebooks'))
import deck
import torch
import matplotlib.pyplot as plt
from PIL import Image
from gradcam.models import load_model, preprocess
from gradcam.saliency import saliency
from gradcam.viz import scale_for_display, gray_rgb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("vgg16", device=device)
model.eval()
print("Ambiente carregado para Beat 3 (Variantes de ReLU).")

## Comparando os mapas com as 3 regras de ReLU

Calculando lado a lado: Vanilla, Deconv e Guided.

In [ ]:
base_dir = os.path.abspath('.')
if os.path.basename(base_dir) == 'notebooks':
    img_path = "assets/cat_dog.jpg"
else:
    img_path = "notebooks/assets/cat_dog.jpg"

pil_img = Image.open(img_path).convert("RGB")
tensor_img = preprocess(pil_img).to(device)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=110)
modes = ["vanilla", "deconv", "guided"]
titles = ["Vanilla Backprop", "Deconvolution", "Guided Backprop"]

for ax, mode, title in zip(axes, modes, titles):
    smap, class_idx = saliency(model, tensor_img, class_idx=None, mode=mode)
    smap_scaled = scale_for_display(smap, high=deck.DISPLAY_HIGH, gamma=deck.DISPLAY_GAMMA)
    ax.imshow(gray_rgb(smap_scaled))
    ax.axis("off")
    ax.set_title(title)

deck.show(fig)

## Punchline: Discriminação de Classe (Spearman)

Lendo os resultados pré-computados de discriminação de classe (`class_discrimination.json`).

In [ ]:
metrics = deck.result("class_discrimination")
print("Métricas de Discriminação de Classe carregadas:")
for method, vals in metrics.items():
    print(f"  - {method}: Spearman = {vals}")